# Fake / Bot Comment Detector

Fine-tunes `distilroberta-base` as a binary classifier (**GENUINE** vs **BOT_OR_SPAM**) and runs it on sample social-media-style comments.

**What was broken, and what was fixed (latest passes):**

9. **Model collapsed to always predicting `BOT_OR_SPAM`, ~100% confidence, regardless of input.** Root cause: training mixed two datasets under one label — the YouTube Spam Collection (real spam vs. real ham comments) and HC3 (human vs. ChatGPT *answers*). ChatGPT answers are almost always long, polished, grammatically complete text, so within the HC3 half of the data, "well-formed sentence" correlated near-perfectly with label 1. That's an easier, more consistent signal to learn than the actual spam-content signal from the YouTube half, so the model latched onto it as a shortcut — and then flagged any articulate, grammatically complete comment as `BOT_OR_SPAM`, genuine or not. **Fix:** HC3 has been removed from training entirely; the model now trains only on the YouTube Spam Collection, where the label genuinely means spam-vs-not. A hand-labelled probe set spanning both classes is now checked automatically after training (Section 10) — the notebook refuses to save the model if it collapses to a single class, so this failure mode gets caught immediately instead of requiring manual inspection.

10. **Training data for the `GENUINE` class was drawn entirely from one narrow slice of the world: YouTube reaction/tutorial comments.** That's still a source-diversity shortcut waiting to happen - a model trained only on "YouTube comment vs. YouTube spam" can end up keying off topic/platform cues (comment section slang, video-reaction phrasing) rather than genuine spam-vs-not signal, and would likely mishandle genuine text from other contexts (reviews, forum posts, etc.). **Fix:** added the UCI **Sentiment Labelled Sentences** dataset (id 331) as a second, independent source of `GENUINE` (label 0) text - 3,000 real, short, human-written sentences from IMDB movie reviews, Amazon product reviews, and Yelp restaurant reviews. It's informal-register text like the YouTube comments (not uniformly polished like ChatGPT output was), but spans completely different topics and platforms, which broadens what `GENUINE` looks like without reintroducing the formality shortcut that HC3 caused. Only the *text* is used - the dataset's own positive/negative sentiment label is discarded entirely, since it has nothing to do with spam-vs-not. The loader was verified end-to-end (both the `ucimlrepo` path and the CSV-mirror fallback) before being wired into the training pipeline, and the amount mixed in is capped to match the existing YouTube `ham` count so it diversifies the `GENUINE` class without swamping the `ham:spam` ratio and creating a new shortcut in the other direction (see Section 2.1).

11. **`GENUINE` was still dominated by YouTube ham, which is almost entirely comments on music/video content** - even after Section 2.1 added the Sentiment Labelled Sentences data, it was capped to match the YouTube ham count 1:1, so video/song-topic comments still made up roughly half of `GENUINE`. That's enough for the model to keep partly relying on "is this a comment about a video/song" as a proxy for genuine, and then treat ordinary genuine comments on unrelated topics (product reviews, workplace chat, forum posts, etc.) as suspicious. **Fix:** the Sentiment Labelled Sentences data is now used in full (uncapped, ~3,000 rows) instead of being capped down to match YouTube ham, and YouTube ham itself is downsampled by 50% (`YT_HAM_KEEP_FRACTION` in Section 2) before combining, so video/song-topic comments are a clear minority of `GENUINE` rather than half of it. The post-training probe set (Section 10) was also diversified with `GENUINE` examples that have no music/video framing at all, so a recurrence of this specific topic shortcut would get caught automatically instead of only showing up on real traffic.

**Earlier fixes (still correct, kept as-is):**

1. **Syntax error / dead code** — the data-loading cell had an incomplete `try/except Exception` block (no `except` body, function never finished, and never called). Rewritten as a single, complete, callable loader.
2. **Broken dataset URL** — `github.com/marshallyin/YouTube-Spam-Collection` returns `404`. Replaced with the official **UCI ML Repository** YouTube Spam Collection (via `ucimlrepo`, dataset id 380), with a working GitHub-hosted CSV mirror as an automatic fallback if UCI's servers are unreachable.
3. **Stale/contradicting documentation** — earlier markdown described a `sms_spam` dataset and a DeBERTa-v3 tokenizer bug (`use_fast=False`) that no longer apply: the pipeline trains `distilroberta-base` on YouTube comment data, which doesn't have that tokenizer bug.
4. **Docs/code mismatch in training hyperparameters** — the markdown promised `learning_rate=5e-6` with a `warmup_ratio`, but `TrainingArguments` had `1e-6` and a hardcoded `warmup_steps=167` (which silently breaks if the dataset size changes). Code now matches the documented settings.
5. **No pinned dependency versions** — `Trainer(processing_class=...)` and `TrainingArguments(eval_strategy=...)` only exist on `transformers>=4.46`. Added a pinned `pip install` cell.
6. **Unnecessary Google Drive mount** — removed; nothing in the pipeline reads from or writes to Drive.

**Fixes carried over from earlier passes (still correct, kept as-is):**
- A `NaNGuardCallback` that fails loudly mid-training instead of silently saving a broken (`NaN`) model.
- A pre-training sanity check (base checkpoint weights are finite) and a post-training sanity check (fine-tuned weights are finite) before anything gets saved.
- Gradient clipping (`max_grad_norm=1.0`).

**Caveat on the data (updated):** the `BOT_OR_SPAM` class still comes entirely from the YouTube Spam Collection, which is real but small (~1,950 comments from 5 videos) and specific to YouTube-style spam (subscribe-bait, channel-promo links) - it won't cover every kind of bot/fake-comment pattern (e.g. review-bombing, coordinated inauthentic praise on other platforms). The `GENUINE` class now spans two sources (YouTube ham + Sentiment Labelled Sentences), which helps, but is still not exhaustive. Treat this as a solid baseline; swap in a larger, platform-matched bot/spam-comment dataset for the `BOT_OR_SPAM` side for production use - and if you add any further auxiliary dataset to either class, keep checking that its label semantics genuinely align with "spam/bot" vs. "genuine" rather than a proxy like "AI-generated" or "formal register", or a shortcut-learning failure like the HC3 one will likely recur.


## 1. Setup, imports & reproducibility

In [1]:
# Pinned versions so this notebook behaves the same way every time it's run.
# transformers>=4.46 is required for Trainer(processing_class=...) and TrainingArguments(eval_strategy=...).
!pip install -q "transformers>=4.46,<4.57" "datasets>=2.19,<3.1" "accelerate>=0.34" "scikit-learn>=1.3" "ucimlrepo>=0.0.7"

import random
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    TrainerCallback,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 99.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.7/472.7 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 72.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.9.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


## 2. Data

Trains on two sources, combined into a single `GENUINE` (0) vs. `BOT_OR_SPAM` (1) corpus:

1. The **YouTube Spam Collection** (UCI ML Repository, id 380) - ~1,950 real YouTube comments labelled spam/ham. Loaded via `ucimlrepo` (the maintained, official way to fetch UCI datasets), with a GitHub-hosted CSV mirror as a fallback if UCI's servers can't be reached from the current environment. Both the ham and spam labels from this dataset are used as-is.
2. The **Sentiment Labelled Sentences** dataset (UCI ML Repository, id 331) - 3,000 real, short, human-written sentences from IMDB, Amazon, and Yelp reviews. Only used to supplement the `GENUINE` class (see Section 2.1 for why, and how much of it gets mixed in); its own positive/negative sentiment label is discarded.

An earlier version of this notebook also mixed in `Hello-SimpleAI/HC3` (human vs. ChatGPT answers) as a second source of "genuine vs. fake" text. That caused the model to collapse to always predicting `BOT_OR_SPAM` - see the changelog above for why. It's been removed; the two sources above are used instead, and both keep the label genuinely meaning spam/bot-vs-not rather than a proxy like formality or AI-authorship.

The combined data is shuffled and split 85/15 into train/validation sets, **stratified by label** so both splits keep the same `GENUINE`/`BOT_OR_SPAM` ratio.


In [2]:
from ucimlrepo import fetch_ucirepo

YOUTUBE_SPAM_CSV_FALLBACK = (
    "https://raw.githubusercontent.com/YBI-Foundation/Dataset/main/YouTube%20Spam.csv"
)


def load_youtube_spam_dataset() -> pd.DataFrame:
    """Loads the UCI YouTube Spam Collection, preferring the official ucimlrepo
    client and falling back to a GitHub-hosted CSV mirror if that's unreachable.
    Returns a DataFrame with 'text' and 'label' columns (1 = spam, 0 = ham).
    """
    try:
        print("Loading YouTube Spam Collection via ucimlrepo (UCI id 380)...")
        yt_repo = fetch_ucirepo(id=380)
        df = pd.DataFrame({
            "text": yt_repo.data.features["CONTENT"],
            "label": yt_repo.data.targets["CLASS"].astype(int),
        }).dropna()
        print(f"Successfully loaded {len(df)} samples from UCI via ucimlrepo.")
        return df
    except Exception as e:
        print(f"ucimlrepo fetch failed ({e!r}); falling back to CSV mirror...")
        df = pd.read_csv(YOUTUBE_SPAM_CSV_FALLBACK)
        df = df.rename(columns={"CONTENT": "text", "CLASS": "label"})
        df = df[["text", "label"]].dropna()
        df["label"] = df["label"].astype(int)
        print(f"Successfully loaded {len(df)} samples from the CSV mirror.")
        return df


### 2.1 Extra `GENUINE` diversity: Sentiment Labelled Sentences (UCI id 331)

Training `GENUINE` on YouTube ham alone risks a narrower version of the same shortcut-learning problem that HC3 caused. Concretely: the YouTube ham comments are almost all replies to music/video content ("love this song", "great tutorial", etc.), so a model trained on them can end up learning "genuine = comments about a video/song" instead of "isn't spam/bot content" - and then wrongly treat ordinary genuine comments on unrelated topics as suspicious.

The Sentiment Labelled Sentences dataset is a good source to mix in because it's:

- **Real, human-written text** - not synthetic or LLM-generated, so it doesn't reintroduce an AI-vs-human proxy.
- **Informal-register**, like the YouTube comments (movie/product/restaurant reviews, not polished prose) - so the model isn't just learning "formal = fake" in a new guise.
- **Topically unrelated to YouTube** (IMDB/Amazon/Yelp reviews vs. video comments) - which directly targets the music/video-topic shortcut described above.

Only the sentence text is used; the dataset's own positive/negative sentiment score is irrelevant here and is discarded. Loaded the same defensive way as the YouTube data: `ucimlrepo` first, with a GitHub/Hugging-Face-hosted CSV mirror as a fallback - both paths were checked end-to-end before being wired into training.

**Balancing (updated):** an earlier version of this cell capped the Sentiment Labelled Sentences addition to match the YouTube ham count 1:1, which kept music/video-topic comments as roughly half of `GENUINE` - not diverse enough to break the topic shortcut. Now:

- The full Sentiment Labelled Sentences set (~3,000 rows) is used, uncapped.
- The YouTube ham count is *downsampled* (via `YT_HAM_KEEP_FRACTION`) before combining, so music/video comments are a clear minority of `GENUINE` rather than half of it, while still being present so the model sees some genuine comments in the same style as the spam it's contrasted against.


In [3]:
SENTIMENT_SENTENCES_CSV_FALLBACK_FILES = [
    "https://huggingface.co/datasets/IsaacDev/sentiment-labelled-sentences/resolve/main/amazon_cells_labelled.txt",
    "https://huggingface.co/datasets/IsaacDev/sentiment-labelled-sentences/resolve/main/imdb_labelled.txt",
    "https://huggingface.co/datasets/IsaacDev/sentiment-labelled-sentences/resolve/main/yelp_labelled.txt",
]


def load_sentiment_labelled_sentences_dataset() -> pd.DataFrame:
    """Loads the UCI Sentiment Labelled Sentences dataset (id 331): 3,000 real, short, human-written
    sentences from IMDB movie reviews, Amazon product reviews, and Yelp restaurant reviews.

    Used here purely as extra GENUINE text - the dataset's own positive/negative sentiment score is
    discarded, since only "is this real, human-written text" matters for this task, not sentiment.
    Prefers ucimlrepo (id 331); falls back to a CSV mirror (three tab-separated .txt files, one per
    source site) if that's unreachable. Returns a DataFrame with a single 'text' column.
    """
    try:
        print("Loading Sentiment Labelled Sentences via ucimlrepo (UCI id 331)...")
        sls_repo = fetch_ucirepo(id=331)
        # Grab the sentence column positionally rather than by name, since this dataset's exact
        # feature-column naming isn't documented the same way as the YouTube one.
        texts = sls_repo.data.features.iloc[:, 0]
        df = pd.DataFrame({"text": texts}).dropna()
        print(f"Successfully loaded {len(df)} samples from UCI via ucimlrepo.")
        return df
    except Exception as e:
        print(f"ucimlrepo fetch failed ({e!r}); falling back to CSV mirror...")
        parts = []
        for url in SENTIMENT_SENTENCES_CSV_FALLBACK_FILES:
            # quoting=3 (QUOTE_NONE): these files contain raw, un-escaped quote characters, so normal
            # CSV quote-handling misparses rows. Tab-separated, no header, two columns per line.
            part = pd.read_csv(
                url, sep="\t", header=None, names=["text", "_sentiment"],
                quoting=3, on_bad_lines="skip",
            )
            parts.append(part[["text"]])
        df = pd.concat(parts, ignore_index=True).dropna()
        print(f"Successfully loaded {len(df)} samples from the CSV mirror (3 files).")
        return df


In [6]:
from datasets import ClassLabel, Dataset

yt_df = load_youtube_spam_dataset()
yt_df["source"] = "youtube_spam_collection"
print("YouTube label balance:", yt_df["label"].value_counts().to_dict())

sls_df = load_sentiment_labelled_sentences_dataset()
sls_df["label"] = 0  # GENUINE - only used as diverse, informal-register human text
sls_df["source"] = "sentiment_labelled_sentences"
print(f"Sentiment Labelled Sentences: {len(sls_df)} examples (used in full, uncapped).")

# YouTube ham comments are almost all replies to music/video content ("love this song", "great
# tutorial"). Left at full size, they would still dominate GENUINE even after adding the ~3,000
# Sentiment Labelled Sentences rows, and the model could keep learning "genuine = comment about a
# video/song" instead of "isn't spam/bot content" - then flag ordinary genuine comments on unrelated
# topics as suspicious. Downsampling YouTube ham (instead of just upsampling the other source) makes
# sure music/video-topic comments are a clear minority of GENUINE, not roughly half of it.
YT_HAM_KEEP_FRACTION = 0.5
yt_ham = yt_df[yt_df["label"] == 0]
yt_spam = yt_df[yt_df["label"] == 1]
yt_ham_downsampled = yt_ham.sample(frac=YT_HAM_KEEP_FRACTION, random_state=SEED)
print(f"Downsampling YouTube ham: {len(yt_ham)} -> {len(yt_ham_downsampled)} "
      f"(kept {YT_HAM_KEEP_FRACTION:.0%}, YouTube spam left untouched at {len(yt_spam)}).")

combined_df = pd.concat([yt_ham_downsampled, yt_spam, sls_df], ignore_index=True)
print("Combined label balance:", combined_df["label"].value_counts().to_dict())
print("Combined source x label breakdown:")
print(combined_df.groupby(["source", "label"]).size())

full_dataset = Dataset.from_pandas(combined_df, preserve_index=False).shuffle(seed=SEED)

# Convert 'label' column to ClassLabel type for stratification
label_features = ClassLabel(num_classes=2, names=["GENUINE", "BOT_OR_SPAM"])
full_dataset = full_dataset.cast_column("label", label_features) # Use cast_column for type conversion

split_dataset = full_dataset.train_test_split(test_size=0.15, seed=SEED, stratify_by_column="label")

train_data = split_dataset["train"]
val_data = split_dataset["test"]
print(f"Total training examples: {len(train_data)}, Validation examples: {len(val_data)}")
print("Train label balance:", pd.Series(train_data["label"]).value_counts().to_dict())
print("Val label balance:", pd.Series(val_data["label"]).value_counts().to_dict())

Loading YouTube Spam Collection via ucimlrepo (UCI id 380)...
Successfully loaded 1956 samples from UCI via ucimlrepo.
YouTube label balance: {1: 1005, 0: 951}
Loading Sentiment Labelled Sentences via ucimlrepo (UCI id 331)...
ucimlrepo fetch failed (DatasetNotFoundError('"Sentiment Labelled Sentences" dataset (id=331) exists in the repository, but is not available for import. Please select a dataset from this list: https://archive.ics.uci.edu/datasets?skip=0&take=10&sort=desc&orderBy=NumHits&search=&Python=true')); falling back to CSV mirror...
Successfully loaded 3000 samples from the CSV mirror (3 files).
Sentiment Labelled Sentences: 3000 examples (used in full, uncapped).
Downsampling YouTube ham: 951 -> 476 (kept 50%, YouTube spam left untouched at 1005).
Combined label balance: {0: 3476, 1: 1005}
Combined source x label breakdown:
source                        label
sentiment_labelled_sentences  0        3000
youtube_spam_collection       0         476
                          

Casting the dataset:   0%|          | 0/4481 [00:00<?, ? examples/s]

Total training examples: 3808, Validation examples: 673
Train label balance: {0: 2954, 1: 854}
Val label balance: {0: 522, 1: 151}


## 3. Tokenizer & tokenization

This pipeline fine-tunes `distilroberta-base`, not `deberta-v3-small` — so the fast/slow tokenizer mismatch that used to cause
`NaN` divergence on DeBERTa-v3 (a known conversion bug between its SentencePiece vocabulary and the Rust `tokenizers`
implementation) doesn't apply here. `distilroberta-base` uses a byte-level BPE tokenizer with no such known issue, so the fast
tokenizer (the default) is used as-is.


In [7]:
MODEL_NAME = "distilroberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_function(examples):
    # Truncate to 128 tokens since social comments are generally short
    return tokenizer(examples["text"], truncation=True, max_length=128)

tokenized_train = train_data.map(preprocess_function, batched=True)
tokenized_val = val_data.map(preprocess_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/3808 [00:00<?, ? examples/s]

Map:   0%|          | 0/673 [00:00<?, ? examples/s]

## 4. Evaluation metrics

In [8]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions, average="binary", zero_division=0),
        "recall": recall_score(labels, predictions, average="binary", zero_division=0),
        "f1": f1_score(labels, predictions, average="binary", zero_division=0),
    }


## 5. Model setup

In [9]:
id2label = {0: "GENUINE", 1: "BOT_OR_SPAM"}
label2id = {"GENUINE": 0, "BOT_OR_SPAM": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)


model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## 6. Pre-training sanity check

Confirm the freshly-loaded checkpoint has no non-finite weights before we spend any time training on top of it.


In [10]:
bad_params_before = [n for n, p in model.named_parameters() if not torch.isfinite(p).all()]
print("Non-finite parameters before training:", bad_params_before)
assert not bad_params_before, "Base checkpoint already contains non-finite weights — do not proceed."


Non-finite parameters before training: []


## 7. NaN guard callback

Some transformer checkpoints are known to occasionally diverge to `NaN` mid-training. This callback stops training
immediately and loudly if that happens, instead of silently saving a broken model.


In [11]:
class NaNGuardCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs and not np.isfinite(logs["loss"]):
            raise RuntimeError(
                f"Training loss became non-finite ({logs['loss']}) at step {state.global_step}. "
                "Stopping before a corrupted model gets saved. Try lowering learning_rate further "
                "(e.g. 1e-6), increasing warmup_ratio, or reducing max_length."
            )


## 8. Training arguments

`learning_rate=5e-6` with `warmup_ratio=0.1` (instead of a hardcoded step count, which silently stops making sense if the
dataset size changes). Gradient clipping stays on as a second line of defense.


In [12]:
training_args = TrainingArguments(
    output_dir="./fake-comment-detector",
    learning_rate=5e-6,
    warmup_ratio=0.1,
    max_grad_norm=1.0,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    fp16=False,
    bf16=False,
)


## 9. Trainer & fine-tuning

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[NaNGuardCallback()],
)

trainer.train()


/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.084900,0.085550,0.973254,0.946309,0.933775,0.940000
2,0.050600,0.084986,0.982169,0.960265,0.960265,0.960265
3,0.066100,0.091093,0.977712,0.935897,0.966887,0.951140


TrainOutput(global_step=714, training_loss=0.13136535607987926, metrics={'train_runtime': 171.4537, 'train_samples_per_second': 66.63, 'train_steps_per_second': 4.164, 'total_flos': 239670780624576.0, 'train_loss': 0.13136535607987926, 'epoch': 3.0})

## 10. Post-training sanity checks & save

Three checks must pass before anything gets saved:
1. All fine-tuned weights are finite (belt-and-braces on top of the `NaNGuardCallback`).
2. Validation metrics look sane.
3. **Non-collapse check**: the model must not predict the same class for every example in a small, hand-labelled probe set
   that spans both classes and, on the `GENUINE` side, deliberately includes topics unrelated to videos/music (restaurant,
   product, workplace, forum-style comments) as well as YouTube-tutorial-style ones. This is exactly the check that would
   have caught the earlier "everything is `BOT_OR_SPAM`" collapse, and would similarly catch a model that only recognizes
   video/song-topic comments as genuine — if it fires, the underlying issue is almost always a spurious shortcut in the
   training data (e.g. label correlating with formality, length, or topic rather than actual spam content), not a
   numerical-stability problem.


In [14]:
bad_params_after = [n for n, p in model.named_parameters() if not torch.isfinite(p).all()]
if bad_params_after:
    raise RuntimeError(
        f"Model has non-finite weights in {len(bad_params_after)} parameter(s) "
        f"(e.g. {bad_params_after[0]}) after training - refusing to save a broken model."
    )
print("Weight check passed: all parameters are finite.")

eval_metrics = trainer.evaluate()
print("Final validation metrics:", eval_metrics)

# Hand-labelled probe set spanning both classes, independent of the training data.
# If the model collapses to predicting one class for all of these, something in the
# training data is teaching it a shortcut instead of actual spam/bot signal.
# Deliberately includes GENUINE examples with no music/video-comment framing at all (restaurant,
# product, workplace, forum-style), since YouTube ham comments are almost all about videos/songs - if
# GENUINE were learned as "comment about a video/song" instead of "isn't spam/bot content", these are
# exactly the examples that would get wrongly flagged, and this check would catch it before saving.
PROBE_COMMENTS = [
    ("I really enjoyed the camera work around the 3-minute mark, great tutorial.", 0),
    ("Has anyone tried doing this step on Ubuntu 24.04? Getting a permissions error.", 0),
    ("This didn't work for me, I get a null pointer exception on line 12.", 0),
    ("Great explanation, finally something that makes sense!", 0),
    ("The service was slow tonight but the pasta more than made up for it.", 0),
    ("Battery life is disappointing compared to the previous model, otherwise solid build quality.", 0),
    ("Our team missed the sprint deadline again because of the flaky CI pipeline.", 0),
    ("Didn't love the ending of this book but the pacing in the middle third was excellent.", 0),
    ("Check my bio for free crypto giveaways and cash rewards! \U0001F680\U0001F4B0", 1),
    ("Invest $100 and make $5000 in 24 hours with Mrs. Johnson! Message her on WhatsApp +1-800-...", 1),
    ("SUBSCRIBE TO MY CHANNEL FOR A FREE IPHONE GIVEAWAY, CLICK THE LINK IN MY BIO", 1),
    ("first!!! who else is early, check out my channel too", 1),
]

model.eval()
device = next(model.parameters()).device
probe_texts = [c for c, _ in PROBE_COMMENTS]
probe_labels = [l for _, l in PROBE_COMMENTS]
probe_inputs = tokenizer(probe_texts, truncation=True, max_length=128, padding=True, return_tensors="pt").to(device)
with torch.no_grad():
    probe_logits = model(**probe_inputs).logits
probe_preds = probe_logits.argmax(dim=-1).tolist()

for text, true_label, pred_label in zip(probe_texts, probe_labels, probe_preds):
    print(f"[true={id2label[true_label]:11s} pred={id2label[pred_label]:11s}] {text}")

distinct_preds = set(probe_preds)
if len(distinct_preds) < 2:
    raise RuntimeError(
        f"Model collapsed to a single predicted class ({[id2label[p] for p in distinct_preds]}) on a "
        "hand-labelled probe set that spans both classes - refusing to save. This is usually a spurious "
        "shortcut in the training data (e.g. label correlating with formality/length/source rather than "
        "actual spam content), not a numerical-stability issue. Inspect the training data sources and "
        "label balance before retrying."
    )

probe_accuracy = sum(p == l for p, l in zip(probe_preds, probe_labels)) / len(probe_labels)
print(f"Non-collapse check passed ({len(distinct_preds)} distinct classes predicted). "
      f"Probe-set accuracy: {probe_accuracy:.2f} (a rough sanity signal, not a formal metric).")

model.save_pretrained("./saved_detector")
tokenizer.save_pretrained("./saved_detector")


Weight check passed: all parameters are finite.


Final validation metrics: {'eval_loss': 0.08498606830835342, 'eval_accuracy': 0.9821693907875185, 'eval_precision': 0.9602649006622517, 'eval_recall': 0.9602649006622517, 'eval_f1': 0.9602649006622517, 'eval_runtime': 1.5149, 'eval_samples_per_second': 444.251, 'eval_steps_per_second': 28.385, 'epoch': 3.0}
[true=GENUINE     pred=GENUINE    ] I really enjoyed the camera work around the 3-minute mark, great tutorial.
[true=GENUINE     pred=GENUINE    ] Has anyone tried doing this step on Ubuntu 24.04? Getting a permissions error.
[true=GENUINE     pred=GENUINE    ] This didn't work for me, I get a null pointer exception on line 12.
[true=GENUINE     pred=GENUINE    ] Great explanation, finally something that makes sense!
[true=GENUINE     pred=GENUINE    ] The service was slow tonight but the pasta more than made up for it.
[true=GENUINE     pred=GENUINE    ] Battery life is disappointing compared to the previous model, otherwise solid build quality.
[true=GENUINE     pred=GENUINE    ] 

('./saved_detector/tokenizer_config.json',
 './saved_detector/special_tokens_map.json',
 './saved_detector/vocab.json',
 './saved_detector/merges.txt',
 './saved_detector/added_tokens.json',
 './saved_detector/tokenizer.json')

## 11. Try it out

In [15]:
import torch
from transformers import pipeline

device_idx = 0 if torch.cuda.is_available() else -1

detector = pipeline(
    "text-classification",
    model="./saved_detector",
    tokenizer="./saved_detector",
    top_k=None,  # return scores for both classes instead of just the top one
    device=device_idx,
)

sample_comments = [
    "Check my bio for free crypto giveaways and cash rewards! \U0001F680\U0001F4B0",
    "I really enjoyed the camera work around the 3-minute mark, great tutorial.",
    "Invest $100 and make $5000 in 24 hours with Mrs. Johnson! Message her on WhatsApp +1-800-...",
    "Has anyone tried doing this step on Ubuntu 24.04? Getting a permissions error.",
]

results = detector(sample_comments)
for comment, scores in zip(sample_comments, results):
    best = max(scores, key=lambda s: s["score"])
    print(f"[{best['label']} ({best['score']:.2f})] -> {comment}")


Device set to use cuda:0


[BOT_OR_SPAM (1.00)] -> Check my bio for free crypto giveaways and cash rewards! 🚀💰
[GENUINE (1.00)] -> I really enjoyed the camera work around the 3-minute mark, great tutorial.
[BOT_OR_SPAM (1.00)] -> Invest $100 and make $5000 in 24 hours with Mrs. Johnson! Message her on WhatsApp +1-800-...
[GENUINE (1.00)] -> Has anyone tried doing this step on Ubuntu 24.04? Getting a permissions error.


In [16]:
new_sample_comments = [
    "Congratulations on your new project! This looks really promising. I'm excited to see more.",
    "Limited time offer! Click the link in my bio to win a free vacation! Don't miss out!",
    "I tried the recipe from your blog and it turned out amazing! Thanks for sharing.",
    "Hey, I found this cool website for crypto trading. Check it out: bit.ly/cryptodeal",
    "This video brightened my day, your content is always so uplifting!",
    "Your account has been compromised. Verify your details here: phishing.site/security-alert",
    "Just finished reading this book, highly recommend it for anyone interested in AI ethics.",
    "Free V-bucks for Fortnite players! Go to freevbux.xyz now before it's too late!"
]

new_results = detector(new_sample_comments)
for comment, scores in zip(new_sample_comments, new_results):
    best = max(scores, key=lambda s: s["score"])
    print(f"[{best['label']} ({best['score']:.2f})] -> {comment}")

[GENUINE (1.00)] -> Congratulations on your new project! This looks really promising. I'm excited to see more.
[BOT_OR_SPAM (1.00)] -> Limited time offer! Click the link in my bio to win a free vacation! Don't miss out!
[GENUINE (0.99)] -> I tried the recipe from your blog and it turned out amazing! Thanks for sharing.
[BOT_OR_SPAM (1.00)] -> Hey, I found this cool website for crypto trading. Check it out: bit.ly/cryptodeal
[GENUINE (1.00)] -> This video brightened my day, your content is always so uplifting!
[BOT_OR_SPAM (0.99)] -> Your account has been compromised. Verify your details here: phishing.site/security-alert
[GENUINE (1.00)] -> Just finished reading this book, highly recommend it for anyone interested in AI ethics.
[BOT_OR_SPAM (0.99)] -> Free V-bucks for Fortnite players! Go to freevbux.xyz now before it's too late!


In [17]:
from google.colab import drive
import os

drive.mount('/content/drive')

drive_save_path = '/content/drive/MyDrive/fake-comment-detector-drive'
os.makedirs(drive_save_path, exist_ok=True)

model.save_pretrained(drive_save_path)
tokenizer.save_pretrained(drive_save_path)

print(f"Model and tokenizer saved to: {drive_save_path}")

Mounted at /content/drive
Model and tokenizer saved to: /content/drive/MyDrive/fake-comment-detector-drive
